### Tool calling
- connection between tool and LLM
- Tool binding 
- LLM knows all the available tools
- LLM know what input to expect

In [2]:
# Model calling and intial setup
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_openai import AzureChatOpenAI
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser 
import warnings
warnings.filterwarnings("ignore") 

load_dotenv()
# Load env
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
AZURE_BASE_URL = os.getenv("AZURE_BASE_URL")
AZURE_OPENAI_API_KEY = os.getenv("AZURE_OPENAI_API_KEY")
AZURE_CHAT_DEPLIOYMENT_NAME = os.getenv("AZURE_CHAT_DEPLIOYMENT_NAME")

parser = StrOutputParser()

llm_gemini = ChatGoogleGenerativeAI(model="gemini-2.0-flash" , api_key= GOOGLE_API_KEY)

llm_openai = AzureChatOpenAI(
    model="gpt-4o-mini",                         
    deployment_name=AZURE_CHAT_DEPLIOYMENT_NAME ,  # deployment name in Azure
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=AZURE_BASE_URL,
    api_version="2024-02-01"
    )
llm_openai.invoke("What are your creater, also what type of LLM are you").content
# llm_gemini.invoke("who is father of india").content

'I was created by OpenAI, an artificial intelligence research organization. I am based on the GPT-3.5 architecture, which is a type of large language model (LLM). My main function is to understand and generate human-like text based on the input I receive. If you have any specific questions or need assistance, feel free to ask!'

In [12]:
from langchain_core.tools import tool

# Creating a tool
@tool 
def multiply(a : int , b :int)->int:
    """fucntion to add two numbers"""
    return a * b

results = multiply.invoke({"a" : 3 , "b":4})

multiply.args

{'a': {'title': 'A', 'type': 'integer'},
 'b': {'title': 'B', 'type': 'integer'}}

In [17]:
# Now tool binding
llm_with_multiply_tool =  llm_gemini.bind_tools([multiply])

#### Tool Calling

In [ ]:
# when LLM need to call a tool with 
results = llm_with_multiply_tool.invoke("Can you multliply 8 with 10 ")

In [25]:
results.tool_calls[0] 

{'name': 'multiply',
 'args': {'a': 8.0, 'b': 10.0},
 'id': 'd3a606e2-a6ca-41e3-ba55-768e591b9e63',
 'type': 'tool_call'}

#### Tool Execution
- LLM doesn't run the tool, it only give the schema to run a tool
- tool execution is on the person using the tool

In [26]:
# Actual python fucntion, that run using args suggested by the LLM
multiply.invoke(results.tool_calls[0].get("args"))

80

In [ ]:
# Tool message
multiply.invoke(results.tool_calls[0] )

# In output we get the ToolMessage that we can again put in the LLM

ToolMessage(content='80', name='multiply', tool_call_id='d3a606e2-a6ca-41e3-ba55-768e591b9e63')

## Creating a currency convertor agent

In [ ]:
# Getting the API Key ready
import requests
from dotenv import load_dotenv
load_dotenv()

responce = requests.get(f"https://v6.exchangerate-api.com/v6/{os.getenv("CURRENCY_API_KEY")}/latest/USD")
responce.json().get("conversion_rates")

In [66]:
# Creaing tools
from langchain_core.tools import InjectedToolArg
from typing import Annotated

@tool
def get_conversion_factor(base_curr : str , target_curr:str)->float:
    """this fucntion convert fetch the conversion factor betweeen base_curr and target_curr """
    
    api_key = os.getenv("CURRENCY_API_KEY")
    responce = requests.get(f"https://v6.exchangerate-api.com/v6/{api_key}/pair/{base_curr}/{target_curr}")
    
    if responce.status_code == 200:
        return responce.json().get("conversion_rate")
    else:
        return None
    
@tool
def convert(base_amount: float , conversion_factor : Annotated[float , InjectedToolArg])->float:
    """Given a amount base currency and conversion factor retun amount in traget currency""" 
    
    return base_amount*conversion_factor

In [67]:
convert.invoke({"base_amount" : 1000 , "conversion_factor" : get_conversion_factor.invoke({"base_curr":"INR"  , 'target_curr' : "USD"})})

11.26

In [99]:
# Binding the tools
from langchain_core.messages import HumanMessage, AIMessage


llm_openai_with_tool = llm_openai.bind_tools([get_conversion_factor , convert])

# testing the llm
message = [HumanMessage("What is the conversion factor between inr to usd and give this conversion factor convert 1000rs to USD")]
results = llm_openai_with_tool.invoke(message)
results

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_d56pAq9zYE6QrmB7QcRYVClB', 'function': {'arguments': '{"base_curr": "inr", "target_curr": "usd"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_En0kxK3HlS4iYFyny8GRMPug', 'function': {'arguments': '{"base_amount": 1000}', 'name': 'convert'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 115, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CMyVElwSQZ8GhxhmbnyU70vsm12ci', 'service_tier': None, 'prompt_filter_results': [{'prompt_index': 0, 'content_filter_results': {'hate': {'filtered': False, 'severity': 'safe'}, 'jailbreak': {'filtered': False, 'detected': F

In [100]:
message.append(results)

In [91]:
results.tool_calls

[{'name': 'get_conversion_factor',
  'args': {'base_curr': 'inr', 'target_curr': 'usd'},
  'id': 'call_vyHPCFjzqw16yQWDg84Xn4eZ',
  'type': 'tool_call'},
 {'name': 'convert',
  'args': {'base_amount': 1000},
  'id': 'call_4N7s8XupwNTHVik3rrSiqzam',
  'type': 'tool_call'}]

In [101]:
for tool_call in results.tool_calls:
    # execute first tool and get conversion rate
    if tool_call.get("name") == 'get_conversion_factor':
        converstion_rate_msg = get_conversion_factor.invoke(tool_call)
        message.append(converstion_rate_msg)
    
    convertion_rate = float(converstion_rate_msg.content)
    
    if tool_call.get("name") == 'convert':
        tool_call.get("args")['conversion_factor'] = convertion_rate  
        convert_tool_msg = convert.invoke(tool_call)
        message.append(convert_tool_msg)


In [102]:
message

[HumanMessage(content='What is the conversion factor between inr to usd and give this conversion factor convert 1000rs to USD', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_d56pAq9zYE6QrmB7QcRYVClB', 'function': {'arguments': '{"base_curr": "inr", "target_curr": "usd"}', 'name': 'get_conversion_factor'}, 'type': 'function'}, {'id': 'call_En0kxK3HlS4iYFyny8GRMPug', 'function': {'arguments': '{"base_amount": 1000}', 'name': 'convert'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 54, 'prompt_tokens': 115, 'total_tokens': 169, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_efad92c60b', 'id': 'chatcmpl-CMyVElwSQZ8GhxhmbnyU70vsm12ci', 'service_tier': 

In [104]:
llm_gemini.invoke(message).content

'The conversion factor between INR to USD is approximately 0.01126.\n\nTherefore, 1000 INR is approximately equal to 11.26 USD.'